In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import mpl_axes_aligner

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------

def pca_func(pca_data, title_suffix="") -> PCA:
    """
    Perform PCA on the given data and plot the explained variance.
    Returns the fitted PCA object.
    """
    pca = PCA()
    pca_out = pca.fit_transform(pca_data)
    
    # Create a new figure for this plot
    plt.figure(figsize=(6, 4))
    plt.plot(pca.explained_variance_, marker='o')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance')
    plt.title(f'PCA Explained Variance{title_suffix}')
    plt.tight_layout()
    
    # Print explained variance summary to console
    summary = pd.DataFrame({
        'Explained Variance Ratio': pca.explained_variance_ratio_,
        'Cumulative Explained Variance': pca.explained_variance_ratio_.cumsum()
    })
    print(f"\nExplained Variance Summary{title_suffix}")
    print(summary)
    
    return pca, pca_out

In [ ]:
# Biplot

def biplot(df_scores: pd.DataFrame, df_loadings: pd.DataFrame, pca: PCA,  labels: pd.Series = None, title: str = "Biplot") -> None:
    """
    Create a PCA biplot with score points and variable loadings.
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Scatter plot for the scores
    ax.scatter(df_scores.PC1.values, df_scores.PC2.values, color='b', alpha=0.7)
    
    # Axis labeling
    expl_var = 100 * pca.explained_variance_ratio_
    ax.set_xlabel(f"PC1 ({expl_var[0]:.1f}% explained var.)", fontsize=10)
    ax.set_ylabel(f"PC2 ({expl_var[1]:.1f}% explained var.)", fontsize=10)
    
    # Determine which labels to use
    if labels is None:
        labels_to_use = df_scores.index
    else:
        labels_to_use = labels
    
    # Label sample points
    for i, name in enumerate(labels_to_use):
        x = df_scores.iloc[i, 0]
        y = df_scores.iloc[i, 1]
        ax.text(x, y, str(name), fontsize=9, color='blue', alpha=0.8)
        
    # Secondary axes for loadings
    ax2 = ax.twinx().twiny()
    
    font = {'color': 'g', 'weight': 'bold', 'size': 10}
    
    # Plot loading vectors and labels
    for col in df_loadings.columns.values:
        tipx = df_loadings.loc['PC1', col]
        tipy = df_loadings.loc['PC2', col]
        ax2.arrow(0, 0, tipx, tipy, color='r', alpha=.5)
        ax2.text(tipx * 1.07, tipy * 1.07, col, fontdict=font, ha='center', va='center')
    
    # Align axes centers
    mpl_axes_aligner.align.xaxes(ax, 0, ax2, 0, 0.5)
    mpl_axes_aligner.align.yaxes(ax, 0, ax2, 0, 0.5)
    
    # Keep square aspect ratio for accurate geometry
    ax.set_aspect('equal', adjustable='datalim')
    ax2.set_aspect('equal', adjustable='datalim')
    
    plt.title(title)
    plt.tight_layout()


In [ ]:
# Load data
dfr = pd.read_csv('data/JamesBond.csv')  # Original raw data frame
df = dfr.iloc[:, 1:]                      # remove non-numeric first column
print(dfr.head())

In [ ]:
# Standardize the data
dfs = StandardScaler().fit_transform(df)

# PCA on standardized data
pca, pca_out = pca_func(dfs, title_suffix=" (Scaled Data)")


In [ ]:
# Prepare PCA scores and loadings
df_scores = pd.DataFrame(pca_out, columns=[f'PC{i}' for i in range(1, df.shape[1] + 1)])
df_loadings = pd.DataFrame(pca.components_, columns=df.columns, index=df_scores.columns)


In [ ]:

# Create biplot
biplot(df_scores, df_loadings, pca, labels=dfr['title'], title="James Bond Dataset - PCA Biplot, Scaled Data")